In [1]:
# standard library imports 
import ast
# third-party imports
import pandas as pd 

In [2]:
morphology = {
    "person": { # 0 
        '1': 'first person',
        '2': 'second person',
        '3': 'third person',
        'x': 'uncertain person'
    },
    "number": { # 1
        's': 'singular',
        'd': 'dual',
        'p': 'plural',
        'x': 'uncertain number'
    },
    "tense": { # 2
        'p': 'present',
        'i': 'imperfect',
        'r': 'perfect',
        's': 'resultative',
        'a': 'aorist',
        'u': 'past',
        'l': 'pluperfect',
        'f': 'future',
        't': 'future perfect',
        'x': 'uncertain tense'
    },
    "mood": { # 3 
        'i': 'indicative',
        's': 'subjunctive',
        'm': 'imperative',
        'o': 'optative',
        'n': 'infinitive',
        'p': 'participle',
        'd': 'gerund',
        'g': 'gerundive',
        'u': 'supine',
        'x': 'uncertain mood',
        'y': 'finiteness unspecified',
        'e': 'indicative or subjunctive',
        'f': 'indicative or imperative',
        'h': 'subjunctive or imperative',
        't': 'finite'
    },
    "voice": { # 4
        'a': 'active',
        'm': 'middle',
        'p': 'passive',
        'e': 'middle or passive',
        'x': 'unspecified'
    },
    "gender": { # 5
        'm': 'masculine',
        'f': 'feminine',
        'n': 'neuter',
        'p': 'masculine or feminine',
        'o': 'masculine or neuter',
        'r': 'feminine or neuter',
        'q': 'masculine, feminine or neuter',
        'x': 'uncertain gender'
    },
    "case": { # 6 
        'n': 'nominative',
        'a': 'accusative',
        'o': 'oblique',
        'g': 'genitive',
        'c': 'genitive or dative',
        'e': 'accusative or dative',
        'd': 'dative',
        'b': 'ablative',
        'i': 'instrumental',
        'l': 'locative',
        'v': 'vocative',
        'x': 'uncertain case',
        'z': 'no case'
    },
    "degree": { # 7
        'p': 'positive',
        'c': 'comparative',
        's': 'superlative',
        'x': 'uncertain degree',
        'z': 'no degree'
    },
    "strength": { # 8 
        'w': 'weak',
        's': 'strong',
        't': 'weak or strong'
    },
    "inflection": { # 9 
        'n': 'non-inflecting',
        'i': 'inflecting'
    }
}

In [3]:
df = pd.read_csv("OUTPUTS/dataframe_02_6.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string"}
)

def drop_unnamed_columns(df):
    """
    Drop columns whose names start with 'Unnamed', typically created
    when saving/loading DataFrames with an index in CSV format.
    """
    return df.loc[:, ~df.columns.str.startswith("Unnamed")]

df = drop_unnamed_columns(df)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_55389/163975699.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_6.csv",


In [4]:
# create sub dataframe consisting of only verbs

df_verbs = df[
    # keep only verbs 
    (df['POS'] == 'V-') 
    # remove auxiliary verbs 
    # as main verbs already cointain info on its corresponding aux verbs
    & (df["Is_Compound_Aux"]==False)]
# amount of verb forms 
verb_count = len(df_verbs)
verb_count

41525

In [5]:
# file type of the columns "col" is simple string 
# to access elements, convert the columns in list "col" to an actual list 
cols = [
    "Compound_Aux_Forms",
    "Compound_Aux_IDs",
    "Compound_Aux_Lemmas",
    "Compound_Aux_Morphology"
]

for col in cols:
    df_verbs[col] = df_verbs[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_55389/3979599355.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_verbs[col] = df_verbs[col].apply(


In [6]:
# assert that all columns starting with "Compound..." contain values of type "list" 
for col in df_verbs:
    if col.startswith("Compound"):
        col_type = df_verbs[col].apply(type).value_counts()
        #print(col_type)
        assert (df_verbs[col].apply(lambda x: isinstance(x, list))).all()

In [7]:
rows = slice(0, 3)
cols = list(range(3,7)) + list(range(13, 15)) + list(range(-7, 0))

df_verbs.iloc[rows, cols]

,Sentence ID,Token ID,Form,Lemma,Morphology,Head ID,Sentence_Text,Is_Compound_Main,Is_Compound_Aux,Compound_Aux_Forms,Compound_Aux_IDs,Compound_Aux_Lemmas,Compound_Aux_Morphology
5,189407,2157778,дьржа,дьржати,-sppamn-si,2157784.0,дьржа землю (рѹськѹ),False,False,[],[],[],[]
11,189407,2157784,повелѣлъ,повелѣти,-sspamn-si,NaN,"азъ ѥсмь повелѣлъ ѿдати бѹицѣ (и (съ, съ, съ))",True,False,[ѥсмь],[2157785],[быти],[1spia----i]
16,189407,2157789,ѿдати,отъдати,--pna----i,2157784.0,"ѿдати бѹицѣ (и (съ, съ, съ))",False,False,[],[],[],[]


In [8]:
# make sure all Compound_Aux verbs have been removed
assert not df_verbs['Is_Compound_Aux'].any()

In [16]:
def df_filter_verbs(df):

    df.loc[
        (df["Is_Compound_Aux"==True]), 
        df["Text Title"]
    ]
        
    return df

df_filter_verbs(df_verbs)

KeyError: False